In [ ]:
# Import and initialize poolparty
import poolparty as pp 
pp.init()

template_pool = pp.from_seq('TCCCGACT<cre>GGAAAGCGGGCAGTGAGCACACAGG</cre>ATTACGG<bc/>AGATCGGA')\
    .named('template_pool').stylize('cre', style='red')

mutated_pool = template_pool.stylize('cre', style='goldenrod')\
                            .mutagenize('cre',
                                        mutation_rate=0.1, 
                                        style_mutations='yellow bold',
                                        mode='random', 
                                        num_states=5, 
                                        prefix='mut_').named('mutated_pool')

deletion_pool = template_pool.stylize('cre', style='salmon')\
                             .deletion_scan('cre', 
                                            deletion_length=5, 
                                            positions=slice(None, None, 5), 
                                            mode='sequential', 
                                            style_deletion='red bold',
                                            prefix='del_').named('deletion_pool')

sites_pool=pp.from_seqs(['AAAAA','TTTTT'], 
                        mode='sequential', 
                        iter_order=-1,
                        prefix='site_').named('sites_pool')

insertion_pool = template_pool.stylize('cre', style='blue')\
                              .insertion_scan('cre', 
                                              ins_pool=sites_pool, 
                                              positions=slice(None,None,5), 
                                              replace=True, 
                                              mode='sequential',
                                              prefix='ins_',
                                              prefix_position='pos_', 
                                              prefix_insertion='site_',
                                              style_insertion='cyan bold').named('insertion_pool')

combo_pool = pp.stack([mutated_pool, deletion_pool, insertion_pool]).named('stack_pool')\
    .repeat_states(2, prefix='v_', iter_order=-2).named('repeated_pool')\
    .insert_kmers('bc', mode='random', length=5, prefix='bc_', style_kmers='green')\
    .named('combo_pool').stylize(which='tags', style='gray')

combo_pool.print_library(show_name=True,seed=12)

pool[15]: seq_length=None, num_states=40
state  name                          seq
    0  mut_0.v_0.bc_0                TCCCGACT<cre>GGACAACGGCAAGCGAGCACACAGC</cre>ATTACGG<bc>AGGCC</bc>AGATCGGA
    1  mut_0.v_1.bc_1                TCCCGACT<cre>GGACAACGGCAAGCGAGCACACAGC</cre>ATTACGG<bc>TGCGG</bc>AGATCGGA
    2  mut_1.v_0.bc_2                TCCCGACT<cre>GCAGAGCGGGCACGGAGCACACAGG</cre>ATTACGG<bc>TTCGG</bc>AGATCGGA
    3  mut_1.v_1.bc_3                TCCCGACT<cre>GCAGAGCGGGCACGGAGCACACAGG</cre>ATTACGG<bc>TTGCA</bc>AGATCGGA
    4  mut_2.v_0.bc_4                TCCCGACT<cre>GGAACACGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>CGCGG</bc>AGATCGGA
    5  mut_2.v_1.bc_5                TCCCGACT<cre>GGAACACGGGCAGTGAGCACACAGG</cre>ATTACGG<bc>GCGCA</bc>AGATCGGA
    6  mut_3.v_0.bc_6                TCCCGACT<cre>GGAAAGCGGGCAGTAAGTACACCGG</cre>ATTACGG<bc>ATATG</bc>AGATCGGA
    7  mut_3.v_1.bc_7                TCCCGACT<cre>GGAAAGCGGGCAGTAAGTACACCGG</cre>ATTACGG<bc>TGGAG</bc>AGATCGGA
    8  mut_4.v_0.bc_8         

Pool(id=15, name='pool[15]', op='op[15]:stylize', num_states=40)

In [2]:
combo_pool.state.print_dag()

pool[13].state (o=-2, n=40)
├── op[12]:get_kmers.state (o=-2, n=40)
├── combo_pool.state (o=-2, n=40)
└── repeated_pool.state (o=-2, n=40)
    └── [op=Product]
        ├── op[11]:repeat.state (o=-2, n=2)
        └── stack_pool.state (o=-1, n=20)
            └── [op=Stack]
                ├── mutated_pool.state (o=0, n=5)
                │   └── [op=Product]
                │       ├── op[3]:mutagenize.state (o=0, n=5)
                │       └── pool[2].state (o=0, n=1)
                │           ├── template_pool.state (o=0, n=1)
                │           └── pool[1].state (o=0, n=1)
                ├── deletion_pool.state (o=0, n=5)
                │   └── [op=Product]
                │       ├── pool[5].state (o=0, n=1)
                │       ├── op[4]:deletion_scan(marker_scan).state (o=0, n=5)
                │       └── pool[1].state (o=0, n=1)
                └── insertion_pool.state (o=-1, n=10)
                    └── [op=Product]
                        ├── sites_pool.sta

In [3]:
combo_pool.generate_library?

Signature: combo_pool.generate_library(**kwargs) -> Union[pandas.core.frame.DataFrame, list[str]]
Docstring:
Generate sequences from this pool.

This is a thin wrapper around poolparty.generate_library().
See that function for full documentation of parameters.
File:      ~/github/poolparty-statetracker/poolparty/src/poolparty/pool.py
Type:      method